# Slug Reclassification

Recomputes slug penetration using clean country denominators (excluding dissolved,
micro, failed, exclusion countries). Classifies slugs by UN region/subregion vectors.
Filters pool for re-clustering.

**Depends on:** `p01_01_country_missingness.ipynb` (must be run first, flags saved)

**Phase 1: Model Definition**

In [1]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

╔══════════════════════════════════════════════════════════════════════════╗
║ QoG METADATA JOINING - PHASE 0 LOADED                                ║
╚══════════════════════════════════════════════════════════════════════════╝

Quick Start:
    metadata = join_metadata()              # Run full pipeline (single isomorphism check)
    metadata = join_metadata_with_cascade()  # Run cascade (strictest → loosest), then union on slug
    quick_check()                             # Diagnostic check
    inspect_exceptions()                      # Review configuration
    show_usage()                              # Detailed documentation

Pipeline Steps:
    1. ingest_and_normalize()            # Load & normalize sources (PDF = qog_slugs_temporal.csv; min_year/max_year ingested)
    2. align_id_variables!(...)          # Harmonize ID vars
    3. run_isomorphism_cascade(...)      # Strictest → loosest until success; returns (stata_df, pdf_df, arrow_df) for union on slug
    4. unify_and_join(..

In [2]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs")

✓ Checksum verified: data/qog_std_ts_jan25_aug.arrow
✓ Loaded: 12391 rows × 2014 cols from data/qog_std_ts_jan25_aug.arrow
  ggis_rowid unique: ✓ | Required columns: ✓ | Missing regions: 0 ✓
Loaded: 12391 rows, 2010 slugs


## Step 1: Load Missingness Results

Re-run country missingness to get the status classifications.

In [3]:
miss_result = run_country_missingness(df, meta_df; verbose=false)
println("Country statuses loaded: $(nrow(miss_result.status)) country-years")
sort(combine(groupby(miss_result.status, :country_status), nrow => :count), :count, rev=true)

Country statuses loaded: 12369 country-years


Row,country_status,count
,String,Int64
1,strong,7408
2,reporting,3189
3,nascent,990
4,microstate,562
5,self_exclusion,78
6,political_exclusion,52
7,failed,42
8,dissolved,28
9,collision,17


## Step 2: Run Reclassification Pipeline

In [6]:
reclass = run_slug_reclassification(df, meta_df, miss_result.status)

Step 1 — Clean Population Universe
    Total country-years: 12391
    Clean country-years: 11590 (excluded 796)
    Clean countries (latest year): 181
    Manual slug exclusions loaded: 21 from data/slug_exclusions.csv

Step 2 — Revised Slug Penetration
    Data end: 2024
    Active window: 2017-2019
    Legacy window: last 5 years before death
    Kept: 1466 slugs
    Dropped: 544 slugs
      experimental         355
      historical           168
      manual:sparse_regional 12
      manual:historical_research 4
      manual:single_country 4
      manual:single_year   1
    Revised globals (≥0.95): 543

Step 3 — UN Region & Subregion Penetration Vectors
    Slugs to process: 1466
    UN Regions: 5
    UN Subregions: 17
    partial         670 slugs
    global          543 slugs
    subregional     226 slugs
    sparse          14 slugs
    regional        13 slugs

Step 4 — Clustering Pool
    Kept slugs (from Step 2): 1466
    Removed:
      global          543
      subregional    

(clean_universe = (clean_rows = 11590×5 DataFrame
   Row │ ident_ccode  ident_year  wpp_pop     ggis_un_subregion_code  ident_cc ⋯
       │ Int64?       Int64       Float64?    Int64                   String   ⋯
───────┼────────────────────────────────────────────────────────────────────────
     1 │           4        1946  missing                         34  AFG      ⋯
     2 │           4        1947  missing                         34  AFG
     3 │           4        1948  missing                         34  AFG
     4 │           4        1949  missing                         34  AFG
     5 │           4        1950     7726.59                      34  AFG      ⋯
     6 │           4        1951     7825.77                      34  AFG
     7 │           4        1952     7932.91                      34  AFG
     8 │           4        1953     8042.65                      34  AFG
     9 │           4        1954     8150.74                      34  AFG      ⋯
    10 │           4

## Slug Disposition Summary

How many slugs in each category?

In [7]:
# Dropped vs kept
dropped = filter(r -> !ismissing(r.drop_reason), reclass.penetration)
kept = filter(r -> ismissing(r.drop_reason), reclass.penetration)
println("Dropped: $(nrow(dropped))")
println(sort(combine(groupby(dropped, :drop_reason), nrow => :count), :count, rev=true))
println("\nKept: $(nrow(kept))")
println(sort(combine(groupby(kept, :temporal_profile), nrow => :count), :count, rev=true))

Dropped: 544
6×2 DataFrame
 Row │ drop_reason                 count 
     │ String?                     Int64 
─────┼───────────────────────────────────
   1 │ experimental                  355
   2 │ historical                    168
   3 │ manual:sparse_regional         12
   4 │ manual:historical_research      4
   5 │ manual:single_country           4
   6 │ manual:single_year              1

Kept: 1466
4×2 DataFrame
 Row │ temporal_profile  count 
     │ String            Int64 
─────┼─────────────────────────
   1 │ modern              879
   2 │ current             238
   3 │ legacy              234
   4 │ anchor              115


## UN Geographic Classification

In [8]:
sort(combine(groupby(reclass.vectors, :un_geo_classification), nrow => :count), :count, rev=true)

Row,un_geo_classification,count
,String,Int64
1,partial,670
2,global,543
3,subregional,226
4,sparse,14
5,regional,13


## Global Slugs (Revised)

Slugs crossing the 95% population-weighted penetration threshold with the clean denominator.

In [7]:
globals = filter(r -> r.un_geo_classification == "global", reclass.vectors)
println("Revised global slugs: $(nrow(globals))")
# Show by prefix
joined = leftjoin(globals[:, [:slug]], meta_df[:, [:slug, :prefix]], on=:slug)
prefix_counts = sort(combine(groupby(joined, :prefix), nrow => :count), :count, rev=true)
println("\nBy prefix:")
for r in eachrow(prefix_counts)
    println("  $(rpad(r.prefix, 15)) $(r.count)")
end

Revised global slugs: 543

By prefix:
  wdi             114
  iaep            28
  vdem            22
  chisols         20
  cbie            18
  ccp             18
  gain            18
  wbgi            18
  br              17
  wpp             16
  pwt             15
  ciri            14
  who             14
  cbi             13
  wgov            13
  gea             12
  ihme            12
  fi              12
  opri            11
  fh              10
  ident           9
  atop            8
  idf             8
  ross            8
  ef              7
  gle             7
  bmr             6
  h               6
  ht              6
  sai             6
  bicc            4
  dr              4
  gpi             4
  spi             4
  ti              4
  fe              3
  ipu             3
  qar             3
  rd              3
  van             3
  bci             2
  cam             2
  chga            2
  gpcr            2
  ied             2
  lld             2
  top             2
 

## Regional & Subregional Slugs

Slugs with geographically concentrated coverage.

In [8]:
for geo_type in ["regional", "subregional"]
    subset = filter(r -> r.un_geo_classification == geo_type, reclass.vectors)
    println("\n=== $(uppercase(geo_type)) ($(nrow(subset)) slugs) ===")
    joined = leftjoin(subset[:, [:slug]], meta_df[:, [:slug, :prefix, :label]], on=:slug)
    prefix_counts = sort(combine(groupby(joined, :prefix), nrow => :count), :count, rev=true)
    for r in eachrow(first(prefix_counts, 10))
        println("  $(rpad(r.prefix, 15)) $(r.count)")
    end
end


=== REGIONAL (13 slugs) ===
  eu              13

=== SUBREGIONAL (226 slugs) ===
  eu              196
  ess             9
  cri             7
  gtr             7
  evep            4
  ideavt          2
  wwbi            1


## Sparse Slugs

In [9]:
sparse = filter(r -> r.un_geo_classification == "sparse", reclass.vectors)
println("Sparse slugs: $(nrow(sparse))")
if nrow(sparse) > 0
    joined = leftjoin(sparse[:, [:slug]], meta_df[:, [:slug, :prefix, :label]], on=:slug)
    for r in eachrow(first(joined, 20))
        println("  $(rpad(r.slug, 25)) $(r.prefix)  $(r.label)")
    end
end

Sparse slugs: 34
  dev_altv1                 dev  Electoral Volatility - Parties above 1%
  dev_othv1                 dev  Electoral Volatility - Parties below 1%
  dev_regv1                 dev  Electoral Volatility - Parties entering/exiting party system
  dev_tv1                   dev  Electoral Volatility - Total
  eu_headenththab           eu  Dentists, per hundred thousand inhabitants
  eu_headentnr              eu  Dentists, number
  eu_headentp               eu  Dentists, inhabitants per dentist
  eu_heahbedlthabp          eu  Long-term care beds (not psychiatric) in hospitals, inhabitant per bed
  eu_heahbedothhabp         eu  Other beds in hospitals, inhabitants per bed
  eu_resnonpf               eu  Researchers in Non-profits as % of total employment - full-time (Female)
  eu_resnonpt               eu  Researchers in Non-profits as % of total employment - full-time (Total)
  eu_trinlw                 eu  Inland waterways transportation (1000's tonnes)
  gtr_centaxdir1800   

## Clustering Pool

Slugs remaining after removing global, regional, subregional, and sparse.

In [10]:
pool = reclass.clustering_pool
println("Clustering pool: $(length(pool.clustering_pool)) slugs")
println("\nRemoved:")
println(pool.summary)

Clustering pool: 670 slugs

Removed:
4×2 DataFrame
 Row │ removal_reason  count 
     │ String          Int64 
─────┼───────────────────────
   1 │ global            543
   2 │ subregional       226
   3 │ sparse             34
   4 │ regional           13


In [ ]:
# Preview pool slugs by prefix
pool_df = DataFrame(slug = pool.clustering_pool)
joined = leftjoin(pool_df, meta_df[:, [:slug, :prefix, :ggis_temporal_profile]], on=:slug)
prefix_counts = sort(combine(groupby(joined, :prefix), nrow => :count), :count, rev=true)
println("Pool slugs by prefix (top 20):")
for r in eachrow(first(prefix_counts, 20))
    println("  $(rpad(r.prefix, 15)) $(r.count)")
end

## Next: Re-Cluster

Run `p01_03_slug_clustering.ipynb` with the filtered pool from above.

In [ ]:
# Save for next notebook
# CSV.write("data/slug_reclassification.csv", reclass.penetration)
# CSV.write("data/clustering_pool.csv", DataFrame(slug=pool.clustering_pool))
# println("\u2705 Saved")